# Assignment 01：動態爬蟲實戰 — Mangaz 漫畫圖書館

## 學習目標
1. 熟悉以 **Selenium** 模擬真實瀏覽器行為，繞過反爬蟲機制。
2. 練習 **多視窗切換** (`window_handles`) 的操作流程。
3. 掌握 **CSS Selector** 與 **PARTIAL_LINK_TEXT** 兩種元素定位策略。
4. 運用 **顯式等待（WebDriverWait + EC）** 搭配 **無窮迴圈** 實作自動翻頁截圖。

## 操作流程說明
```
Step 1：啟動 Chrome (附反偵測設定)
    ↓
Step 2：開啟漫畫詳情頁，點擊「免費閱讀」按鈕
    ↓
Step 3：切換到新開啟的視窗
    ↓
Step 4：點擊「すぐに読む（立即閱讀）」連結
    ↓
Step 5：自動翻頁並截圖每一頁漫畫（核心迴圈）
    ↓
Step 6：關閉 WebDriver
```

> **注意**：請確認已安裝 `selenium` 套件，且 ChromeDriver 版本與本機 Chrome 相符。

---
## Step 1：匯入套件 & 啟動 WebDriver（附反偵測設定）

### 知識點
| 設定項目 | 說明 |
|---|---|
| `excludeSwitches` | 移除 Chrome 自動化旗標，避免被網站偵測 |
| `useAutomationExtension` | 停用自動化擴充功能 |
| `user-agent` | 偽裝成一般使用者的瀏覽器 |
| `--disable-blink-features=AutomationControlled` | 關閉 Blink 引擎的自動化特徵值 |

### 本作業額外需要匯入的模組
| 模組 | 用途 |
|---|---|
| `WebDriverWait` | 建立顯式等待物件，最多等待 N 秒 |
| `EC` (`expected_conditions`) | 提供各種「等待條件」，如元素可點擊、元素出現等 |
| `TimeoutException` | 超過顯式等待時間後拋出的例外，用於偵測最後一頁 |
| `time` | 提供 `time.sleep()` 讓程式強制等待 N 秒 |

In [ ]:
# TODO 1：匯入 Selenium 基礎套件
# 提示：需要 webdriver、Options、By
# --- 請在此填寫 ---
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# TODO 2：匯入顯式等待相關模組
# 提示：WebDriverWait 來自 selenium.webdriver.support.ui
#       EC (expected_conditions) 來自 selenium.webdriver.support
#       TimeoutException 來自 selenium.common.exceptions
# --- 請在此填寫 ---
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException,ElementNotInteractableException, NoSuchElementException


# TODO 3：匯入 time 模組（用於 time.sleep()）
# --- 請在此填寫 ---
import time

# TODO 4：建立 Options 物件
# --- 請在此填寫 ---
options = Options()

# TODO 5：加入以下三項「反偵測」設定
#   (a) excludeSwitches -> ["enable-automation"]
#   (b) useAutomationExtension -> False
#   (c) user-agent 字串（使用 Windows Chrome 121 的 UA 即可）
# --- 請在此填寫 ---
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36") 

# TODO 6：關閉 Blink 引擎的自動化特徵
#   提示：add_argument('--disable-blink-features=...')
# --- 請在此填寫 ---
options.add_argument("--disable-blink-features=AutomationControlled")

# TODO 7：實例化 WebDriver，並將 options 傳入
# --- 請在此填寫 ---
driver = webdriver.Chrome(options=options)

# TODO 8：開啟目標 URL，並列印頁面標題以確認成功
#   目標網址：https://www.mangaz.com/book/detail/157901
# --- 請在此填寫 ---
driver.get("https://www.mangaz.com/book/detail/157901")
print("Page Title:", driver.title)

Page Title: 【全巻無料】ロボ犬ハチ。 - 永吉武 | 男性向け漫画が読み放題 - マンガ図書館Z


In [3]:
driver.quit()

---
## Step 2：定位並點擊「免費閱讀」按鈕

### 知識點
- 使用 `By.CSS_SELECTOR` 透過 CSS 選擇器精準定位元素。
- 目標元素的 CSS Selector：`button.open-viewer.book-begin.ga`

```
意義拆解：
  button          → 標籤為 <button>
  .open-viewer    → class 包含 open-viewer
  .book-begin     → class 包含 book-begin
  .ga             → class 包含 ga（Google Analytics 追蹤用）
```

In [5]:
# TODO 9：使用 CSS_SELECTOR 定位「免費閱讀」按鈕
#   CSS Selector：'button.open-viewer.book-begin.ga'
# --- 請在此填寫 ---
button = driver.find_element(By.CSS_SELECTOR, 'button.open-viewer.book-begin.ga')

# TODO 10：點擊該按鈕
# --- 請在此填寫 ---
button.click()

---
## Step 3：切換到新開啟的瀏覽器視窗

### 知識點
- 點擊按鈕後，網站通常會**開啟新分頁**，此時 WebDriver 仍停留在原視窗。
- `driver.window_handles` 回傳所有視窗代號的**列表（list）**，順序為開啟先後。
- 使用 `driver.switch_to.window(handle)` 切換至指定視窗。

```python
# 概念示意
all_windows = ['CDwindow-原始視窗', 'CDwindow-新視窗']
#               index 0               index -1 (最後一個)
```

In [6]:
# TODO 11：取得所有視窗的代號（Handle）列表
# --- 請在此填寫 ---
all_windows = driver.window_handles

# TODO 12：切換到最後一個（最新開啟的）視窗
#   提示：使用 driver.switch_to.window(...) 搭配 all_windows 的最後一個元素
# --- 請在此填寫 ---
driver.switch_to.window(all_windows[-1])

---
## Step 4：定位並點擊「すぐに読む（立即閱讀）」連結

### 知識點
- 使用 `By.PARTIAL_LINK_TEXT` 透過**部分連結文字**尋找 `<a>` 標籤元素。
- 適用於連結文字較長，或只知道部分關鍵字的情境。
- 目標文字片段：`'すぐに読む'`

In [7]:
# TODO 13：使用 PARTIAL_LINK_TEXT 定位「すぐに読む」連結元素
# --- 請在此填寫 ---
Read_Now = driver.find_element(By.PARTIAL_LINK_TEXT, "すぐに読む")

# TODO 14：點擊該連結
# --- 請在此填寫 ---
Read_Now.click()


---
## Step 5：自動翻頁並截圖每一頁漫畫（核心迴圈）

### 知識點：顯式等待 vs 隱式等待
| 等待方式 | 方法 | 特性 |
|---|---|---|
| **隱式等待** | `driver.implicitly_wait(N)` | 全域生效，找不到元素時最多等 N 秒 |
| **顯式等待** | `WebDriverWait(driver, N).until(EC.xxx)` | 針對特定條件等待，彈性高、推薦使用 |

### 知識點：`EC.element_to_be_clickable`
- 等待元素**出現在 DOM 中且可被點擊**（`enabled` 且 `visible`）。
- 語法：`EC.element_to_be_clickable((By.CSS_SELECTOR, 'selector字串'))`
- 注意傳入的是**一個 tuple**，外層需加括號。

### 知識點：迴圈終止條件
- 當所有頁面爬取完畢後，**下一頁按鈕不再出現**。
- `WebDriverWait` 超過等待時間會拋出 `TimeoutException`。
- 在 `except TimeoutException` 區塊中執行 `break`，即可優雅地結束迴圈。

```
迴圈邏輯示意：
  ┌─────────────────────────────────┐
  │  while True:                    │
  │    步驟 A：截圖當前可見的圖片   │
  │    步驟 B：嘗試點擊下一頁按鈕   │
  │      ├─ 成功 → sleep(2) → 繼續  │
  │      └─ TimeoutException → break │
  └─────────────────────────────────┘
```

### 目標元素資訊
| 元素 | CSS Selector |
|---|---|
| 漫畫圖片 | `div.page_image img.image` |
| 下一頁按鈕 | `div.flip.flip-left` |

In [8]:
# ── 初始化設定 ──────────────────────────────────────────────────────────────

# TODO 15：建立顯式等待物件，設定最長等待時間為 10 秒
#   提示：WebDriverWait(driver, 秒數)
# --- 請在此填寫 ---
wait_element = WebDriverWait(driver, 10)

# TODO 16：設定全域圖片計數器（初始值為 0）
#   用途：防止翻頁後檔名重複覆蓋
# --- 請在此填寫 ---
total_image_count = 0


# 2. 進入無窮迴圈
while True:

    # ── 步驟 A：擷取當前頁面圖片 ──────────────────────────────────────────

    # TODO 17：使用 find_elements 取得當前頁面所有符合條件的圖片元素
    #   CSS Selector：'div.page_image img.image'
    #   注意：find_elements（複數）回傳串列；find_element（單數）回傳單一元素
    # --- 請在此填寫 ---
    image_elements = driver.find_elements(By.CSS_SELECTOR, 'div.page_image img.image')

    for img_element in image_elements:
        # 只處理目前畫面上「可見」的圖片，避免抓到隱藏元素
        if img_element.is_displayed():
            file_path = f"manga_page_{total_image_count}.png"

            # TODO 18a：對該圖片元素截圖，並儲存到 file_path
            #   提示：Selenium WebElement 本身有 .screenshot(路徑) 方法
            # --- 請在此填寫 ---
            img_element.screenshot(file_path)

            print(f"成功擷取頁面並儲存為: {file_path}")

            # TODO 18b：每成功儲存一張圖，將計數器加 1
            # --- 請在此填寫 ---
            total_image_count += 1

    # ── 步驟 B：嘗試尋找並點擊「下一頁」按鈕 ─────────────────────────────
    try:
        # TODO 18c：使用 wait_element.until(EC.element_to_be_clickable(...))
        #   等待 CSS Selector 為 'div.flip.flip-left' 的元素出現且可點擊
        #   注意：EC.element_to_be_clickable 傳入的是 (By.xxx, '選擇器') 的 tuple
        # --- 請在此填寫 ---
        next_page = wait_element.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'div.flip.flip-left')))

        # TODO 18d：點擊下一頁按鈕
        # --- 請在此填寫 ---
        next_page.click()

        print("已點擊下一頁，等待畫面載入...")

        # TODO 18e：點擊後強制等待 2 秒，確保 DOM 與圖片載入完成
        #   提示：time.sleep(秒數)
        # --- 請在此填寫 ---
        time.sleep(2)

    # ── 步驟 C：Break Condition ────────────────────────────────────────────
    except TimeoutException:
        # 等待超過 10 秒仍找不到下一頁按鈕 → 已到達最後一頁
        print("【系統提示】找不到下一頁按鈕，已達最後一頁，結束爬取迴圈。")
        break

成功擷取頁面並儲存為: manga_page_0.png
已點擊下一頁，等待畫面載入...
成功擷取頁面並儲存為: manga_page_1.png
成功擷取頁面並儲存為: manga_page_2.png
已點擊下一頁，等待畫面載入...
成功擷取頁面並儲存為: manga_page_3.png
成功擷取頁面並儲存為: manga_page_4.png
已點擊下一頁，等待畫面載入...
成功擷取頁面並儲存為: manga_page_5.png
成功擷取頁面並儲存為: manga_page_6.png
已點擊下一頁，等待畫面載入...
成功擷取頁面並儲存為: manga_page_7.png
成功擷取頁面並儲存為: manga_page_8.png
已點擊下一頁，等待畫面載入...
成功擷取頁面並儲存為: manga_page_9.png
成功擷取頁面並儲存為: manga_page_10.png
已點擊下一頁，等待畫面載入...
成功擷取頁面並儲存為: manga_page_11.png
成功擷取頁面並儲存為: manga_page_12.png
已點擊下一頁，等待畫面載入...
成功擷取頁面並儲存為: manga_page_13.png
成功擷取頁面並儲存為: manga_page_14.png
已點擊下一頁，等待畫面載入...
成功擷取頁面並儲存為: manga_page_15.png
成功擷取頁面並儲存為: manga_page_16.png
已點擊下一頁，等待畫面載入...
成功擷取頁面並儲存為: manga_page_17.png
成功擷取頁面並儲存為: manga_page_18.png
已點擊下一頁，等待畫面載入...
成功擷取頁面並儲存為: manga_page_19.png
成功擷取頁面並儲存為: manga_page_20.png
已點擊下一頁，等待畫面載入...
成功擷取頁面並儲存為: manga_page_21.png
成功擷取頁面並儲存為: manga_page_22.png
已點擊下一頁，等待畫面載入...
成功擷取頁面並儲存為: manga_page_23.png
成功擷取頁面並儲存為: manga_page_24.png
已點擊下一頁，等待畫面載入...
成功擷取頁面並儲存為: manga_page_25.png
成功擷取頁面並儲存

---
## Step 6：關閉 WebDriver

### 知識點
- `driver.quit()` → 關閉**所有**瀏覽器視窗，並結束 WebDriver 行程。
- `driver.close()` → 只關閉**當前**視窗，行程仍保持運行。

> 良好習慣：爬蟲結束後務必呼叫 `driver.quit()` 釋放資源。

In [9]:
# TODO 19：關閉所有瀏覽器視窗並結束 WebDriver
# --- 請在此填寫 ---
driver.quit()